In [34]:
import re
import math
import json
import string
import warnings
import collections
warnings.filterwarnings("ignore")

import pdfplumber
import nltk
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

try:
    from langdetect import detect as lang_detect
except ImportError:
    lang_detect = lambda x: "en"

try:
    import textstat
    HAS_TEXTSTAT = True
except ImportError:
    HAS_TEXTSTAT = False

# Download NLTK data
for pkg in ["punkt", "stopwords", "wordnet", "averaged_perceptron_tagger",
            "omw-1.4", "punkt_tab"]:
    try:
        nltk.download(pkg, quiet=True)
    except:
        pass

print("✅ All libraries loaded successfully")

✅ All libraries loaded successfully


In [35]:
# CELL 2 — PDF Upload for VS Code
import tkinter as tk
from tkinter import filedialog
import os

MAX_FEATURES  = 100
TOP_N_WORDS   = 30
DASHBOARD_OUT = "nlp_dashboard.html"

# Open file picker
root = tk.Tk()
root.withdraw()
root.wm_attributes('-topmost', True)

PDF_PATH = filedialog.askopenfilename(
    title="Select your PDF file (100+ pages)",
    filetypes=[("PDF files", "*.pdf")]
)

if PDF_PATH:
    size_mb = os.path.getsize(PDF_PATH) / (1024*1024)
    print(f"✅ File selected: {os.path.basename(PDF_PATH)}")
    print(f"   Full path : {PDF_PATH}")
    print(f"   Size      : {size_mb:.1f} MB")
    print(f"\n   Now run Cell 3 onwards ▶")
else:
    print("❌ No file selected — run this cell again and pick a PDF")

✅ File selected: EN8033_Project Brief-Final-version.pdf
   Full path : C:/Users/admin/Downloads/EN8033_Project Brief-Final-version.pdf
   Size      : 0.9 MB

   Now run Cell 3 onwards ▶


In [36]:
print("=" * 60)
print("Q1(a): PDF READING & TEXT EXTRACTION")
print("=" * 60)

pages_text = []

with pdfplumber.open(PDF_PATH) as pdf:
    total_pages = len(pdf.pages)
    print(f"\n📄 Total number of pages: {total_pages}")

    for i, page in enumerate(pdf.pages):
        text = page.extract_text()
        if text:
            pages_text.append(text.strip())
        else:
            pages_text.append("")

full_text = "\n".join(pages_text)

print(f"📝 Total characters extracted : {len(full_text):,}")
print(f"📝 Total words (raw)          : {len(full_text.split()):,}")
print(f"\n--- Sample extracted text (first 500 chars) ---\n")
print(full_text[:500])
print("\n--- Sample from page 5 ---\n")
print(pages_text[4][:400] if len(pages_text) > 4 else "N/A")

Q1(a): PDF READING & TEXT EXTRACTION

📄 Total number of pages: 12
📝 Total characters extracted : 16,919
📝 Total words (raw)          : 2,540

--- Sample extracted text (first 500 chars) ---

Assessment Cover Sheet
Assessment Title Project: Design of Electric drive to feed 3 phase centrifugal fan (IM)
:
Assessment Type Controlled Group Not must-pass
Due Date 30.05.2026 Course Code EN8033 & EN8033T
Course Title Electric Drives
Internal Moderator’s Name Dr. Zakareya Hassan
External Examiner’s Name Dr. David Cheneler
Instructions:
1. This cover sheet must be completed (section in blue below) and attached to your assessment before
submission in hard copy/soft copy.
2. The time allowed fo

--- Sample from page 5 ---

o Control algorithm implementation (you may implement an idealized controller
using Multisim components or Matlab components
o Protection elements and sensors (must be included in schematic/circuit diagram)
Note:(Not required in Multisim simulation file)
Simulate and capture res

In [37]:
print("\n" + "=" * 60)
print("Q1(b): TEXT PREPROCESSING")
print("=" * 60)

stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

# Track word counts at each stage for funnel chart
stage_counts = {}

# Step 1: Lowercase
text_lower = full_text.lower()
stage_counts["1. Raw"] = len(full_text.split())
print(f"\n✅ Step 1 — Lowercase applied")
print(f"   Sample: '{full_text[:60]}'\n   → '{text_lower[:60]}'")

# Step 2: Remove numbers using Regex
REGEX_NUMBERS  = r"\d+"
text_no_nums   = re.sub(REGEX_NUMBERS, "", text_lower)
nums_removed   = len(re.findall(REGEX_NUMBERS, text_lower))
stage_counts["2. After remove numbers"] = len(text_no_nums.split())
print(f"\n✅ Step 2 — Remove numbers  |  Regex pattern: {REGEX_NUMBERS}")
print(f"   Numbers removed: {nums_removed}")

# Step 3: Remove special symbols using Regex
REGEX_SYMBOLS  = r"[^a-z\s]"
text_no_sym    = re.sub(REGEX_SYMBOLS, " ", text_no_nums)
stage_counts["3. After remove symbols"] = len(text_no_sym.split())
print(f"\n✅ Step 3 — Remove special symbols  |  Regex pattern: {REGEX_SYMBOLS}")

# Step 4: Remove extra spaces using Regex
REGEX_SPACES   = r"\s+"
text_no_space  = re.sub(REGEX_SPACES, " ", text_no_sym).strip()
stage_counts["4. After clean spaces"] = len(text_no_space.split())
print(f"\n✅ Step 4 — Remove extra spaces  |  Regex pattern: {REGEX_SPACES}")

# Step 5: Remove punctuation
text_no_punct  = text_no_space.translate(str.maketrans("", "", string.punctuation))
stage_counts["5. After remove punctuation"] = len(text_no_punct.split())
print(f"\n✅ Step 5 — Remove punctuation")

# Step 6: Tokenize
tokens = word_tokenize(text_no_punct)
stage_counts["6. Tokenized"] = len(tokens)
print(f"\n✅ Step 6 — Tokenization")
print(f"   Total tokens: {len(tokens):,}")
print(f"   Sample tokens: {tokens[:15]}")

# Step 7: Stop word removal
stop_words_found  = [t for t in tokens if t in stop_words]
valid_words       = [t for t in tokens if t not in stop_words and len(t) > 1]
stage_counts["7. After stopword removal"] = len(valid_words)

print(f"\n✅ Step 7 — Stop word removal")
print(f"   Stop words found  : {len(stop_words_found):,}")
print(f"   Valid words left  : {len(valid_words):,}")
print(f"   Sample stop words : {list(set(stop_words_found))[:10]}")
print(f"   Sample valid words: {valid_words[:15]}")

# Step 8: Stemming
stemmed_words = [stemmer.stem(w) for w in valid_words]
stage_counts["8. After stemming"] = len(stemmed_words)
print(f"\n✅ Step 8 — Stemming (Porter Stemmer)")
stem_examples = [(valid_words[i], stemmed_words[i])
                 for i in range(min(10, len(valid_words)))
                 if valid_words[i] != stemmed_words[i]]
print(f"   Examples: {stem_examples}")

# Step 9: Lemmatization
lemmatized_words = [lemmatizer.lemmatize(w) for w in valid_words]
stage_counts["9. After lemmatization"] = len(lemmatized_words)
print(f"\n✅ Step 9 — Lemmatization (WordNet)")
lemma_examples = [(valid_words[i], lemmatized_words[i])
                  for i in range(min(10, len(valid_words)))
                  if valid_words[i] != lemmatized_words[i]]
print(f"   Examples: {lemma_examples}")

# Cleaned text string (for TF-IDF)
cleaned_text = " ".join(valid_words)
print(f"\n📊 Preprocessing Summary:")
for k, v in stage_counts.items():
    print(f"   {k}: {v:,} words")




Q1(b): TEXT PREPROCESSING

✅ Step 1 — Lowercase applied
   Sample: 'Assessment Cover Sheet
Assessment Title Project: Design of E'
   → 'assessment cover sheet
assessment title project: design of e'

✅ Step 2 — Remove numbers  |  Regex pattern: \d+
   Numbers removed: 262

✅ Step 3 — Remove special symbols  |  Regex pattern: [^a-z\s]

✅ Step 4 — Remove extra spaces  |  Regex pattern: \s+

✅ Step 5 — Remove punctuation

✅ Step 6 — Tokenization
   Total tokens: 2,328
   Sample tokens: ['assessment', 'cover', 'sheet', 'assessment', 'title', 'project', 'design', 'of', 'electric', 'drive', 'to', 'feed', 'phase', 'centrifugal', 'fan']

✅ Step 7 — Stop word removal
   Stop words found  : 635
   Valid words left  : 1,631
   Sample stop words : ['an', 'only', 'any', 'your', 'the', 'by', 'no', 'will', 'from', 'i']
   Sample valid words: ['assessment', 'cover', 'sheet', 'assessment', 'title', 'project', 'design', 'electric', 'drive', 'feed', 'phase', 'centrifugal', 'fan', 'im', 'assessment']

✅ S

In [38]:
print("\n" + "=" * 60)
print("Q1(c): FEATURE EXTRACTION — ONE HOT ENCODING")
print("=" * 60)

# Take top 50 unique valid words for OHE display
ohe_words  = valid_words[:200]
unique_vocab = sorted(set(ohe_words))[:30]   # keep display manageable

ohe_matrix = []
for w in ohe_words[:50]:
    row = [1 if w == v else 0 for v in unique_vocab]
    ohe_matrix.append(row)

ohe_df = pd.DataFrame(ohe_matrix, columns=unique_vocab)
ohe_df.insert(0, "Word", ohe_words[:50])

print(f"\nOne Hot Encoding — shape: {ohe_df.shape}")
print(ohe_df.head(10).to_string())


Q1(c): FEATURE EXTRACTION — ONE HOT ENCODING

One Hot Encoding — shape: (50, 31)
         Word  abc  acknowledged  affirm  ai  alawi  allowed  aluminum  assessing  assessment  assessor  assigned  attached  ayman  beyond  blue  brief  byjumon  capacity  carries  centrifugal  cheneler  cilo  cited  code  comments  company  completed  consulted  controlled  cooling
0  assessment    0             0       0   0      0        0         0          0           1         0         0         0      0       0     0      0        0         0        0            0         0     0      0     0         0        0          0          0           0        0
1       cover    0             0       0   0      0        0         0          0           0         0         0         0      0       0     0      0        0         0        0            0         0     0      0     0         0        0          0          0           0        0
2       sheet    0             0       0   0      0        0      

In [39]:
print("\n" + "=" * 60)
print("Q1(c): FEATURE EXTRACTION — TF-IDF")
print("=" * 60)

# Split cleaned text into chunks (treat each chunk as a document)
words_per_chunk = 300
word_list       = cleaned_text.split()
chunks          = [" ".join(word_list[i:i+words_per_chunk])
                   for i in range(0, len(word_list), words_per_chunk)
                   if len(word_list[i:i+words_per_chunk]) > 20]

tfidf_vec = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1,1))
tfidf_mat = tfidf_vec.fit_transform(chunks)

feature_names = tfidf_vec.get_feature_names_out()
tfidf_df      = pd.DataFrame(
    tfidf_mat.toarray(),
    columns=feature_names,
    index=[f"Chunk_{i+1}" for i in range(len(chunks))]
)

print(f"\nTF-IDF Matrix shape: {tfidf_df.shape}")
print(f"Feature names (first 20): {list(feature_names[:20])}")
print(f"\nTF-IDF Values (top 5 chunks, top 10 features):")
print(tfidf_df.iloc[:5, :10].round(4).to_string())

# Get mean TF-IDF per word for plotting
mean_tfidf     = tfidf_df.mean(axis=0).sort_values(ascending=False)
top_tfidf_df   = mean_tfidf.head(TOP_N_WORDS).reset_index()
top_tfidf_df.columns = ["word", "tfidf_score"]
top_tfidf_df["tfidf_score"] = top_tfidf_df["tfidf_score"].round(5)

print(f"\nTop {TOP_N_WORDS} words by mean TF-IDF score:")
print(top_tfidf_df.head(15).to_string())



Q1(c): FEATURE EXTRACTION — TF-IDF

TF-IDF Matrix shape: (6, 100)
Feature names (first 20): ['analysis', 'answered', 'appropriate', 'assessment', 'audience', 'automatic', 'bahraini', 'calculations', 'circuit', 'clear', 'company', 'complete', 'conclusion', 'connection', 'control', 'controller', 'converter', 'cooling', 'correct', 'cover']

TF-IDF Values (top 5 chunks, top 10 features):
         analysis  answered  appropriate  assessment  audience  automatic  bahraini  calculations  circuit   clear
Chunk_1    0.0000       0.0       0.0366       0.555       0.0     0.0000    0.0427        0.0000   0.0000  0.0000
Chunk_2    0.0793       0.0       0.0340       0.000       0.0     0.0000    0.0000        0.0000   0.0000  0.0000
Chunk_3    0.0000       0.0       0.0338       0.000       0.0     0.0000    0.0394        0.1401   0.1183  0.0000
Chunk_4    0.0548       0.0       0.0469       0.000       0.0     0.0000    0.2190        0.1946   0.1095  0.0000
Chunk_5    0.0899       0.0       0.0

In [40]:
print("\n" + "=" * 60)
print("BONUS: BIGRAMS & TRIGRAMS")
print("=" * 60)

from nltk.util import ngrams

bigrams_raw   = list(ngrams(valid_words, 2))
trigrams_raw  = list(ngrams(valid_words, 3))

bigram_freq   = collections.Counter(bigrams_raw).most_common(20)
trigram_freq  = collections.Counter(trigrams_raw).most_common(15)

bigram_df     = pd.DataFrame([(" ".join(k), v) for k,v in bigram_freq],
                             columns=["bigram","count"])
trigram_df    = pd.DataFrame([(" ".join(k), v) for k,v in trigram_freq],
                             columns=["trigram","count"])

print(f"\nTop 10 Bigrams:\n{bigram_df.head(10).to_string()}")
print(f"\nTop 10 Trigrams:\n{trigram_df.head(10).to_string()}")



BONUS: BIGRAMS & TRIGRAMS

Top 10 Bigrams:
                  bigram  count
0      correct partially      9
1         electric drive      6
2           drive system      6
3            tower tower      5
4                lfi lfi      5
5      partially correct      5
6  design specifications      5
7          points points      5
8         project design      4
9            three phase      4

Top 10 Trigrams:
                       trigram  count
0    correct partially correct      5
1            tower tower tower      4
2                  lfi lfi lfi      4
3    circuit correct partially      4
4  partially correct incorrect      4
5  correct partially incorrect      4
6        three phase induction      3
7        phase induction motor      3
8       bahraini saudi british      3
9         daily speed schedule      3


In [41]:
word_freq     = collections.Counter(valid_words)
freq_df       = pd.DataFrame(word_freq.most_common(200),
                             columns=["word","frequency"])
freq_df["rank"] = range(1, len(freq_df)+1)
freq_df["log_rank"] = np.log10(freq_df["rank"])
freq_df["log_freq"] = np.log10(freq_df["frequency"])

In [42]:
compare_words = valid_words[:40]
stem_compare  = [stemmer.stem(w) for w in compare_words]
lemma_compare = [lemmatizer.lemmatize(w) for w in compare_words]

# words that changed under either method
diff_mask = [s != w or l != w
             for w, s, l in zip(compare_words, stem_compare, lemma_compare)]
diff_words  = [w for w, d in zip(compare_words, diff_mask) if d][:20]
diff_stems  = [stemmer.stem(w) for w in diff_words]
diff_lemmas = [lemmatizer.lemmatize(w) for w in diff_words]

stem_lemma_df = pd.DataFrame({
    "original"     : diff_words,
    "stemmed"      : diff_stems,
    "lemmatized"   : diff_lemmas,
    "orig_len"     : [len(w) for w in diff_words],
    "stem_len"     : [len(s) for s in diff_stems],
    "lemma_len"    : [len(l) for l in diff_lemmas],
})
print(f"\nStemming vs Lemmatization (sample):")
print(stem_lemma_df.head(10).to_string())



Stemming vs Lemmatization (sample):
      original    stemmed   lemmatized  orig_len  stem_len  lemma_len
0   assessment     assess   assessment        10         6         10
1   assessment     assess   assessment        10         6         10
2        title       titl        title         5         4          5
3     electric     electr     electric         8         6          8
4  centrifugal  centrifug  centrifugal        11         9         11
5   assessment     assess   assessment        10         6         10
6   controlled    control   controlled        10         7         10
7         pass       pass          pas         4         4          3
8       course      cours       course         6         5          6
9       course      cours       course         6         5          6


In [43]:

print("\n" + "=" * 60)
print("INNOVATION 1: AUTO-GENERATED EXTRACTIVE SUMMARY")
print("=" * 60)

# Score each sentence by how many top TF-IDF words it contains
top_keywords = set(top_tfidf_df["word"].head(20).tolist())  # top 20 TF-IDF words

# Split original full text into sentences
import re as _re
raw_sentences = _re.split(r'(?<=[.!?])\s+', full_text)
raw_sentences = [s.strip() for s in raw_sentences if len(s.split()) > 5]

def score_sentence(sentence, keywords):
    words = sentence.lower().split()
    return sum(1 for w in words if w in keywords)

scored = [(score_sentence(s, top_keywords), s) for s in raw_sentences]
scored.sort(key=lambda x: x[0], reverse=True)

# Pick top 5 unique sentences and restore original order
top5_set = []
seen = set()
for score, sent in scored:
    clean = sent[:60]  # use start as unique key
    if clean not in seen and score > 0:
        top5_set.append(sent)
        seen.add(clean)
    if len(top5_set) == 5:
        break

# Sort back into document order
summary_sentences = sorted(top5_set, key=lambda s: full_text.find(s[:40]))

print("\n📄 Auto-Generated 5-Line Summary (Extractive):\n")
print("-" * 60)
for i, sent in enumerate(summary_sentences, 1):
    # Trim very long sentences for readability
    display = sent if len(sent) <= 200 else sent[:197] + "..."
    print(f"  {i}. {display}")
print("-" * 60)
print(f"\n✅ Summary built from top {len(top_keywords)} TF-IDF keywords")
print(f"   Sentences scored: {len(raw_sentences)}")
print(f"   Top sentence score: {scored[0][0]} keyword matches")



INNOVATION 1: AUTO-GENERATED EXTRACTIVE SUMMARY

📄 Auto-Generated 5-Line Summary (Extractive):

------------------------------------------------------------
  1. In general, cooling tower fans follow the well-known fan laws:
• Fan torque increases with the square of speed: 𝑇 ∝ 𝑁2
• Fan power increases with the cube of speed: 𝑃 ∝ 𝑁3
Because the cooling requi...
  2. • Select one of the below specifications for your design (No two groups can select the
same specifications)
Tower 1 Tower 2 Tower 3 Tower 4 Tower 5 Tower 6
T (N.m) 4.146e-5*N2 1.186e-4*N2 1.019e-3*...
  3. o Control algorithm implementation (you may implement an idealized controller
using Multisim components or Matlab components
o Protection elements and sensors (must be included in schematic/circuit...
  4. Energy Savings from Variable-Speed Electric Drive for Fan Motor:
Prepare a complete engineering study that quantifies the annual electrical energy and
cost savings achieved by using an electric dri...
  5. Prove the des

In [44]:
print("\n" + "=" * 60)
print("INNOVATION 2: KEYWORD CSV EXPORT")
print("=" * 60)

# Build top 50 keywords with frequency AND tfidf score
word_freq_dict = dict(collections.Counter(valid_words))

keyword_export_df = top_tfidf_df.copy()                      # has 'word' and 'tfidf_score'
keyword_export_df = keyword_export_df.head(50)
keyword_export_df["frequency"] = keyword_export_df["word"].map(
    lambda w: word_freq_dict.get(w, 0)
)
keyword_export_df["rank"] = range(1, len(keyword_export_df) + 1)
keyword_export_df = keyword_export_df[["rank", "word", "tfidf_score", "frequency"]]

# Export to CSV
CSV_OUT = "top_keywords.csv"
keyword_export_df.to_csv(CSV_OUT, index=False)

print(f"\n✅ Exported top {len(keyword_export_df)} keywords → {CSV_OUT}")
print(f"\nPreview (top 15):")
print(keyword_export_df.head(15).to_string(index=False))
print(f"\n📁 File saved: {CSV_OUT}")
print(f"   Columns: rank | word | tfidf_score | frequency")



INNOVATION 2: KEYWORD CSV EXPORT

✅ Exported top 30 keywords → top_keywords.csv

Preview (top 15):
 rank       word  tfidf_score  frequency
    1      speed      0.18555         34
    2     points      0.13027         13
    3    correct      0.12429         29
    4     design      0.12295         26
    5      drive      0.11912         20
    6      motor      0.11439         23
    7    control      0.10877         24
    8        fan      0.10103         15
    9  standards      0.09344         13
   10 simulation      0.09277         16
   11 assessment      0.09250          9
   12      tower      0.08849         11
   13    circuit      0.08669         18
   14    project      0.08518         16
   15 protection      0.08294         17

📁 File saved: top_keywords.csv
   Columns: rank | word | tfidf_score | frequency


In [45]:
print("\n" + "=" * 60)
print("INNOVATION 3: READABILITY SCORE WITH LABEL")
print("=" * 60)

# Calculate Flesch Reading Ease (already have textstat imported)
readability_score = 0
if HAS_TEXTSTAT:
    try:
        readability_score = textstat.flesch_reading_ease(full_text)
        grade             = textstat.flesch_kincaid_grade(full_text)
        reading_time_min  = textstat.reading_time(full_text, ms_per_char=14.69)
    except:
        readability_score = 0
        grade             = 0
        reading_time_min  = 0
else:
    # Manual Flesch fallback if textstat not installed
    sentence_count = max(1, len(_re.split(r'[.!?]', full_text)))
    word_count     = max(1, len(full_text.split()))
    syllable_count = sum(
        max(1, len(_re.findall(r'[aeiouAEIOU]', w))) for w in full_text.split()
    )
    readability_score = (
        206.835
        - 1.015 * (word_count / sentence_count)
        - 84.6  * (syllable_count / word_count)
    )
    grade            = 0
    reading_time_min = round(word_count / 200, 1)

# Map score → human label
if readability_score >= 90:
    level = "Very Easy"
    audience = "accessible to a 5th-grade student"
    emoji = "🟢"
elif readability_score >= 70:
    level = "Easy"
    audience = "accessible to a general audience"
    emoji = "🟢"
elif readability_score >= 60:
    level = "Standard"
    audience = "suitable for a general adult reader"
    emoji = "🟡"
elif readability_score >= 50:
    level = "Fairly Difficult"
    audience = "suited to high-school or college level"
    emoji = "🟡"
elif readability_score >= 30:
    level = "Difficult"
    audience = "reads at university / professional level"
    emoji = "🔴"
else:
    level = "Very Difficult"
    audience = "reads at postgraduate / academic level"
    emoji = "🔴"

print(f"""
{emoji}  Flesch Reading Ease Score : {readability_score:.1f}  /  100
    Readability Level         : {level}
    Audience                  : This document {audience}
    Flesch-Kincaid Grade      : {grade}
    Estimated Reading Time    : {reading_time_min} minutes

Score Guide:
    90–100  → Very Easy   (5th grade)
    70–89   → Easy        (General public)
    60–69   → Standard    (Adult reader)
    50–59   → Fairly Difficult (High school / college)
    30–49   → Difficult   (University level)
    0–29    → Very Difficult (Postgraduate / academic)
""")
print(f"✅ Readability analysis complete")



INNOVATION 3: READABILITY SCORE WITH LABEL

🔴  Flesch Reading Ease Score : 14.6  /  100
    Readability Level         : Very Difficult
    Audience                  : This document reads at postgraduate / academic level
    Flesch-Kincaid Grade      : 0
    Estimated Reading Time    : 12.7 minutes

Score Guide:
    90–100  → Very Easy   (5th grade)
    70–89   → Easy        (General public)
    60–69   → Standard    (Adult reader)
    50–59   → Fairly Difficult (High school / college)
    30–49   → Difficult   (University level)
    0–29    → Very Difficult (Postgraduate / academic)

✅ Readability analysis complete


In [46]:
print("\n" + "=" * 60)
print("INNOVATION 4: NLPAnalyzer — REUSABLE CLASS INTERFACE")
print("=" * 60)

class NLPAnalyzer:
    """
    Drop-in reusable NLP pipeline for any PDF.

    Usage:
        analyzer = NLPAnalyzer("your_document.pdf")
        analyzer.run_pipeline()
        analyzer.show_summary()
    """

    def __init__(self, pdf_path):
        self.pdf_path        = pdf_path
        self.full_text       = ""
        self.pages_text      = []
        self.valid_words     = []
        self.top_tfidf_df    = None
        self.summary         = []
        self.readability     = {}
        self._pipeline_done  = False

    # ── Step 1: Extract ──────────────────────────────────────
    def extract(self):
        import pdfplumber
        self.pages_text = []
        with pdfplumber.open(self.pdf_path) as pdf:
            self.total_pages = len(pdf.pages)
            for page in pdf.pages:
                t = page.extract_text()
                self.pages_text.append(t.strip() if t else "")
        self.full_text = "\n".join(self.pages_text)
        return self

    # ── Step 2: Preprocess ───────────────────────────────────
    def preprocess(self):
        import re, string
        from nltk.tokenize import word_tokenize
        from nltk.corpus import stopwords
        text = self.full_text.lower()
        text = re.sub(r"\d+", "", text)
        text = re.sub(r"[^a-z\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        text = text.translate(str.maketrans("", "", string.punctuation))
        tokens = word_tokenize(text)
        sw = set(stopwords.words("english"))
        self.valid_words = [t for t in tokens if t not in sw and len(t) > 1]
        return self

    # ── Step 3: TF-IDF ───────────────────────────────────────
    def tfidf(self):
        from sklearn.feature_extraction.text import TfidfVectorizer
        import numpy as np, pandas as pd
        cleaned = " ".join(self.valid_words)
        chunk_size = 300
        words = cleaned.split()
        chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]
        vec = TfidfVectorizer(max_features=100)
        mat = vec.fit_transform(chunks)
        df  = pd.DataFrame(mat.toarray(), columns=vec.get_feature_names_out())
        mean = df.mean(axis=0).sort_values(ascending=False)
        self.top_tfidf_df = mean.head(50).reset_index()
        self.top_tfidf_df.columns = ["word", "tfidf_score"]
        return self

    # ── Step 4: Summarize ────────────────────────────────────
    def summarize(self, n=5):
        import re
        keywords = set(self.top_tfidf_df["word"].head(20).tolist())
        sents = re.split(r'(?<=[.!?])\s+', self.full_text)
        sents = [s.strip() for s in sents if len(s.split()) > 5]
        scored = sorted(sents, key=lambda s: sum(1 for w in s.lower().split() if w in keywords), reverse=True)
        seen, result = set(), []
        for s in scored:
            key = s[:60]
            if key not in seen:
                result.append(s)
                seen.add(key)
            if len(result) == n:
                break
        self.summary = sorted(result, key=lambda s: self.full_text.find(s[:40]))
        return self

    # ── Step 5: Readability ──────────────────────────────────
    def score_readability(self):
        score = 0
        try:
            import textstat
            score = textstat.flesch_reading_ease(self.full_text)
        except:
            pass
        if score >= 70:   label = "Easy — accessible to a general audience"
        elif score >= 50: label = "Standard — suited to college level"
        elif score >= 30: label = "Difficult — reads at university level"
        else:             label = "Very Difficult — postgraduate / academic"
        self.readability = {"score": round(score, 1), "label": label}
        return self

    # ── Run everything ───────────────────────────────────────
    def run_pipeline(self):
        print(f"🔄 Running full NLP pipeline on: {self.pdf_path}")
        self.extract().preprocess().tfidf().summarize().score_readability()
        self._pipeline_done = True
        print(f"✅ Pipeline complete — {self.total_pages} pages, {len(self.valid_words):,} tokens")
        return self

    # ── Display summary ──────────────────────────────────────
    def show_summary(self):
        if not self._pipeline_done:
            print("⚠️  Run .run_pipeline() first")
            return
        print("\n" + "=" * 60)
        print("NLPAnalyzer — RESULTS SUMMARY")
        print("=" * 60)
        print(f"  Pages          : {self.total_pages}")
        print(f"  Tokens kept    : {len(self.valid_words):,}")
        print(f"  Unique words   : {len(set(self.valid_words)):,}")
        print(f"  Vocab richness : {round(len(set(self.valid_words))/max(1,len(self.valid_words))*100, 1)}%")
        print(f"\n  Readability    : {self.readability['score']} — {self.readability['label']}")
        print(f"\n  Top 10 Keywords by TF-IDF:")
        for _, row in self.top_tfidf_df.head(10).iterrows():
            print(f"    {row['word']:<20} {row['tfidf_score']:.5f}")
        print(f"\n  Auto-Summary (5 sentences):")
        for i, s in enumerate(self.summary, 1):
            display = s if len(s) <= 180 else s[:177] + "..."
            print(f"    {i}. {display}")
        print("=" * 60)

    # ── Export keywords CSV ──────────────────────────────────
    def export_keywords(self, path="keywords.csv"):
        if not self._pipeline_done:
            print("⚠️  Run .run_pipeline() first")
            return
        import collections
        freq = dict(collections.Counter(self.valid_words))
        df = self.top_tfidf_df.copy()
        df["frequency"] = df["word"].map(lambda w: freq.get(w, 0))
        df["rank"] = range(1, len(df) + 1)
        df = df[["rank", "word", "tfidf_score", "frequency"]]
        df.to_csv(path, index=False)
        print(f"✅ Keywords exported → {path}  ({len(df)} rows)")

    # ── Export dashboard ─────────────────────────────────────
    def export_dashboard(self, path="report.html"):
        print(f"ℹ️  Run the dashboard cell (last cell) to generate: {path}")


# ── Demo: run the class on the same PDF ──────────────────────
print("\n🚀 Demonstrating NLPAnalyzer class on your PDF:\n")

analyzer = NLPAnalyzer(PDF_PATH)
analyzer.run_pipeline()
analyzer.show_summary()
analyzer.export_keywords("keywords.csv")

print("""
─────────────────────────────────────────────────────────────
The NLPAnalyzer class makes this tool reusable for ANY PDF:

    analyzer = NLPAnalyzer("any_document.pdf")
    analyzer.run_pipeline()
    analyzer.show_summary()

Anyone can clone this repo, drop in their PDF, and get a
full NLP breakdown in 3 lines. No wiring required.
─────────────────────────────────────────────────────────────
""")


INNOVATION 4: NLPAnalyzer — REUSABLE CLASS INTERFACE

🚀 Demonstrating NLPAnalyzer class on your PDF:

🔄 Running full NLP pipeline on: C:/Users/admin/Downloads/EN8033_Project Brief-Final-version.pdf


✅ Pipeline complete — 12 pages, 1,631 tokens

NLPAnalyzer — RESULTS SUMMARY
  Pages          : 12
  Tokens kept    : 1,631
  Unique words   : 682
  Vocab richness : 41.8%

  Readability    : 0 — Very Difficult — postgraduate / academic

  Top 10 Keywords by TF-IDF:
    speed                0.18555
    points               0.13027
    correct              0.12429
    design               0.12295
    drive                0.11912
    motor                0.11439
    control              0.10877
    fan                  0.10103
    standards            0.09344
    simulation           0.09277

  Auto-Summary (5 sentences):
    1. In general, cooling tower fans follow the well-known fan laws:
• Fan torque increases with the square of speed: 𝑇 ∝ 𝑁2
• Fan power increases with the cube of speed: 𝑃 ∝ 𝑁3
Becau...
    2. • Select one of the below specifications for your design (No two groups can select the
same specifications)
Tower 1 Tower 2 Tower 3 Tower 4 Tower 5 Tower 6
T (N.m) 4.146e-5*N2 1.

In [47]:
try:
    detected_lang = lang_detect(full_text[:2000])
except:
    detected_lang = "en"

readability = 0
if HAS_TEXTSTAT:
    try:
        readability = textstat.flesch_reading_ease(full_text[:5000])
    except:
        readability = 0

vocab_richness = round(len(set(valid_words)) / max(len(valid_words),1) * 100, 2)
stopword_pct   = round(len(stop_words_found) / max(len(tokens),1) * 100, 1)
content_pct    = round(100 - stopword_pct, 1)

print(f"\nDocument Language  : {detected_lang}")
print(f"Readability Score  : {readability}")
print(f"Vocabulary Richness: {vocab_richness}%")
print(f"Stopword ratio     : {stopword_pct}%")
print(f"Content word ratio : {content_pct}%")


Document Language  : en
Readability Score  : 0
Vocabulary Richness: 41.81%
Stopword ratio     : 27.3%
Content word ratio : 72.7%


In [48]:
print("\n" + "=" * 60)
print("INNOVATION: GRAVEYARD & THRONE ROOM DASHBOARD")
print("=" * 60)

# Prepare data for 3D visualization
dead_tokens = []
dead_tokens += [(w, "number (regex \\d+)")     for w in re.findall(r"\b\d+\b", full_text.lower())][:40]
dead_tokens += [(w, "punctuation (regex)")     for w in re.findall(r"[^\w\s]", full_text)][:30]
dead_tokens += [(w, "stop word")               for w in stop_words_found][:60]

alive_tokens = top_tfidf_df.head(25).to_dict("records")

dead_json  = json.dumps(dead_tokens[:80])
alive_json = json.dumps(alive_tokens)
steps_json = json.dumps([
    {"name": k, "count": v} for k, v in stage_counts.items()
])
bigram_json = json.dumps(bigram_df.head(10).to_dict("records"))

# Document stats
doc_stats = {
    "pages"       : total_pages,
    "raw_words"   : len(full_text.split()),
    "unique_words": len(set(valid_words)),
    "valid_words" : len(valid_words),
    "stop_removed": len(stop_words_found),
    "language"    : detected_lang,
    "readability" : round(readability, 1) if HAS_TEXTSTAT else "N/A",
    "vocab_rich"  : vocab_richness,
}
stats_json = json.dumps(doc_stats)

HTML_DASHBOARD = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>NLP Pipeline Dashboard</title>
<script src="https://cdn.jsdelivr.net/npm/three@0.128.0/build/three.min.js"></script>
<script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
<style>
*{{margin:0;padding:0;box-sizing:border-box}}
body{{font-family:'Segoe UI',Arial,sans-serif;background:#0f1117;color:#e0e0e0;overflow-x:hidden}}
h1{{text-align:center;padding:22px 0 4px;font-size:1.7rem;letter-spacing:1px;color:#fff}}
.subtitle{{text-align:center;color:#888;font-size:.9rem;margin-bottom:18px}}
.stats-row{{display:flex;gap:12px;padding:0 20px 16px;flex-wrap:wrap;justify-content:center}}
.stat-card{{background:#1a1d2e;border:1px solid #2a2d3e;border-radius:12px;padding:14px 22px;text-align:center;min-width:130px}}
.stat-num{{font-size:1.6rem;font-weight:700;color:#7c8dff}}
.stat-lbl{{font-size:.75rem;color:#888;margin-top:2px}}
.section{{padding:14px 20px 6px;font-size:.8rem;color:#666;letter-spacing:2px;text-transform:uppercase}}
.two-col{{display:grid;grid-template-columns:1fr 1fr;gap:16px;padding:0 20px 20px}}
.panel{{background:#1a1d2e;border:1px solid #2a2d3e;border-radius:14px;padding:16px}}
.panel h3{{font-size:.95rem;color:#aaa;margin-bottom:10px}}
canvas#world{{width:100%;height:480px;display:block;border-radius:14px;cursor:grab;background:#080c18}}
.world-wrap{{padding:0 20px 20px;position:relative}}
.world-label{{position:absolute;top:24px;left:50%;transform:translateX(-50%);
  background:rgba(0,0,0,.7);color:#fff;padding:6px 18px;border-radius:20px;font-size:.8rem;pointer-events:none}}
.legend{{display:flex;gap:16px;justify-content:center;padding:0 0 8px;font-size:.78rem}}
.dot{{width:10px;height:10px;border-radius:50%;display:inline-block;margin-right:5px;vertical-align:middle}}
.grave-list{{max-height:200px;overflow-y:auto;font-size:.78rem;line-height:1.8}}
.grave-item{{display:flex;justify-content:space-between;padding:2px 6px;border-radius:4px}}
.grave-item:nth-child(odd){{background:#131629}}
.tag{{font-size:.68rem;padding:2px 7px;border-radius:8px;background:#2a1a1a;color:#e07070}}
.throne-item{{display:flex;align-items:center;gap:8px;padding:3px 0;font-size:.8rem}}
.bar-fill{{height:8px;border-radius:4px;background:linear-gradient(90deg,#f0a500,#ffe066)}}
.plotly-graph{{background:#1a1d2e;border-radius:14px;border:1px solid #2a2d3e}}
footer{{text-align:center;padding:20px;color:#444;font-size:.75rem}}
</style>
</head>
<body>
<h1>🧠 NLP Pipeline Dashboard</h1>
<p class="subtitle">Complete analysis · Graveyard &amp; Throne Room · Interactive 3D</p>

<div class="stats-row" id="statsRow"></div>

<div class="section">3D World — Graveyard &amp; Throne Room</div>
<div class="world-wrap">
  <canvas id="world"></canvas>
  <div class="world-label">🪦 Graveyard (left) — removed tokens &nbsp;|&nbsp; 👑 Throne Room (right) — TF-IDF survivors &nbsp;·&nbsp; Drag to rotate</div>
</div>
<div class="legend">
  <span><span class="dot" style="background:#e05555"></span>Removed by number regex</span>
  <span><span class="dot" style="background:#e0a030"></span>Removed by punctuation regex</span>
  <span><span class="dot" style="background:#7070e0"></span>Stop word removed</span>
  <span><span class="dot" style="background:#f0d060"></span>TF-IDF survivor (throne)</span>
</div>

<div class="two-col">
  <div class="panel">
    <h3>🪦 Graveyard — all removed tokens</h3>
    <div class="grave-list" id="graveList"></div>
  </div>
  <div class="panel">
    <h3>👑 Throne Room — top TF-IDF words</h3>
    <div id="throneList"></div>
  </div>
</div>

<div class="two-col">
  <div class="panel plotly-graph" id="funnelDiv"></div>
  <div class="panel plotly-graph" id="bigramDiv"></div>
</div>

<div style="padding:0 20px 20px">
  <div class="panel plotly-graph" id="tfidfDiv"></div>
</div>

<footer>NLP Assignment Dashboard · Built with Python + Plotly + Three.js</footer>

<script>
const DEAD   = {dead_json};
const ALIVE  = {alive_json};
const STAGES = {steps_json};
const STATS  = {stats_json};
const BIGRAMS= {bigram_json};

// ── Stats cards ───────────────────────────────────────────
const statDefs = [
  ["Pages",          STATS.pages,         "#7c8dff"],
  ["Raw words",      STATS.raw_words.toLocaleString(), "#5ecbaa"],
  ["Unique words",   STATS.unique_words.toLocaleString(), "#f0a040"],
  ["Valid tokens",   STATS.valid_words.toLocaleString(), "#60c060"],
  ["Stop removed",   STATS.stop_removed.toLocaleString(), "#e07070"],
  ["Language",       STATS.language.toUpperCase(), "#c080ff"],
  ["Vocab richness", STATS.vocab_rich+"%", "#5ec8ff"],
];
const sr = document.getElementById("statsRow");
statDefs.forEach(([l,v,c])=>{{
  sr.innerHTML += `<div class="stat-card"><div class="stat-num" style="color:${{c}}">${{v}}</div><div class="stat-lbl">${{l}}</div></div>`;
}});

// ── Grave & Throne lists ──────────────────────────────────
const gl = document.getElementById("graveList");
DEAD.forEach(([w,r])=>{{
  gl.innerHTML += `<div class="grave-item"><span>${{w}}</span><span class="tag">${{r}}</span></div>`;
}});

const tl = document.getElementById("throneList");
const maxScore = Math.max(...ALIVE.map(a=>a.tfidf_score));
ALIVE.forEach((a,i)=>{{
  const pct = Math.round(a.tfidf_score/maxScore*100);
  const crown = i===0?"👑":i===1?"🥈":i===2?"🥉":"";
  tl.innerHTML += `<div class="throne-item">
    <span style="min-width:16px;text-align:center">${{crown||"·"}}</span>
    <span style="min-width:100px;color:#ddd">${{a.word}}</span>
    <div style="flex:1"><div class="bar-fill" style="width:${{pct}}%"></div></div>
    <span style="min-width:52px;text-align:right;color:#aaa;font-size:.72rem">${{a.tfidf_score.toFixed(4)}}</span>
  </div>`;
}});

// ── Plotly funnel ─────────────────────────────────────────
Plotly.newPlot("funnelDiv", [{{
  type:"funnel", y:STAGES.map(s=>s.name), x:STAGES.map(s=>s.count),
  textinfo:"value+percent initial",
  marker:{{color:["#378ADD","#1D9E75","#EF9F27","#D85A30","#534AB7","#993556","#0F6E56","#BA7517","#639922"]}}
}}],{{
  title:{{text:"Preprocessing Funnel",font:{{color:"#ccc"}}}},
  paper_bgcolor:"transparent", plot_bgcolor:"transparent",
  font:{{color:"#aaa"}}, margin:{{t:50,b:20,l:10,r:10}}, height:280
}}, {{displayModeBar:false}});

// ── Plotly bigram bar ─────────────────────────────────────
Plotly.newPlot("bigramDiv", [{{
  type:"bar", orientation:"h",
  y:BIGRAMS.map(b=>b.bigram), x:BIGRAMS.map(b=>b.count),
  marker:{{color:"#534AB7"}},
  text:BIGRAMS.map(b=>b.count), textposition:"outside"
}}],{{
  title:{{text:"Top Bigrams",font:{{color:"#ccc"}}}},
  paper_bgcolor:"transparent", plot_bgcolor:"transparent",
  font:{{color:"#aaa"}}, margin:{{t:50,b:20,l:130,r:30}},
  height:280, yaxis:{{autorange:"reversed"}}
}},{{displayModeBar:false}});

// ── Plotly TF-IDF scatter ─────────────────────────────────
Plotly.newPlot("tfidfDiv", [{{
  type:"scatter", mode:"markers+text",
  x:ALIVE.map(a=>a.word), y:ALIVE.map(a=>a.tfidf_score),
  text:ALIVE.map(a=>a.tfidf_score.toFixed(4)), textposition:"top center",
  marker:{{
    size:ALIVE.map(a=>a.tfidf_score*600+10),
    color:ALIVE.map(a=>a.tfidf_score),
    colorscale:"Viridis", showscale:true,
    colorbar:{{title:"TF-IDF",tickfont:{{color:"#aaa"}},titlefont:{{color:"#aaa"}}}}
  }},
  hovertemplate:"<b>%{{x}}</b><br>TF-IDF: %{{y:.5f}}<extra></extra>"
}}],{{
  title:{{text:"TF-IDF Scatter — Required Graph",font:{{color:"#ccc",size:15}}}},
  paper_bgcolor:"transparent", plot_bgcolor:"transparent",
  font:{{color:"#aaa"}},
  xaxis:{{title:"Words",tickangle:-40,color:"#888"}},
  yaxis:{{title:"TF-IDF Score",color:"#888"}},
  margin:{{t:55,b:100,l:70,r:20}}, height:380
}},{{displayModeBar:false}});

// ── THREE.JS Graveyard & Throne Room ─────────────────────
const canvas3 = document.getElementById("world");
const renderer = new THREE.WebGLRenderer({{canvas:canvas3,antialias:true}});
renderer.setPixelRatio(Math.min(devicePixelRatio,2));
renderer.setClearColor(0x080c18);

const scene = new THREE.Scene();
const camera = new THREE.PerspectiveCamera(45,2,0.1,500);
camera.position.set(0,12,32);
camera.lookAt(0,0,0);

const resize3 = ()=>{{
  renderer.setSize(canvas3.clientWidth,canvas3.clientHeight,false);
  camera.aspect=canvas3.clientWidth/canvas3.clientHeight;
  camera.updateProjectionMatrix();
}};
resize3();
new ResizeObserver(resize3).observe(canvas3);

scene.add(new THREE.AmbientLight(0xffffff,0.5));
const dl=new THREE.DirectionalLight(0xffffff,0.9);
dl.position.set(10,20,15); scene.add(dl);
const dl2=new THREE.DirectionalLight(0x8899ff,0.3);
dl2.position.set(-15,5,-10); scene.add(dl2);

// Ground planes
const mkPlane=(color,x)=>{{
  const m=new THREE.Mesh(
    new THREE.PlaneGeometry(24,20),
    new THREE.MeshStandardMaterial({{color,roughness:1,transparent:true,opacity:0.18}})
  );
  m.rotation.x=-Math.PI/2; m.position.set(x,-0.01,0); scene.add(m);
}};
mkPlane(0x2a1010,-14); mkPlane(0x101a2a,14);

// Grid
const grid=new THREE.GridHelper(60,40,0x222233,0x1a1a2a);
grid.position.y=0; scene.add(grid);

// Divider wall
const wall=new THREE.Mesh(
  new THREE.BoxGeometry(0.15,8,20),
  new THREE.MeshStandardMaterial({{color:0x334455,roughness:0.8}})
);
wall.position.set(0,3.5,0); scene.add(wall);

// Make sprite label
function makeSprite(word, color){{
  const lc=document.createElement("canvas");
  lc.width=256; lc.height=64;
  const ctx=lc.getContext("2d");
  ctx.clearRect(0,0,256,64);
  ctx.fillStyle="#"+color.toString(16).padStart(6,"0");
  ctx.font="bold 22px monospace";
  ctx.textAlign="center";
  ctx.fillText(word.length>12?word.slice(0,11)+"…":word,128,44);
  const tex=new THREE.CanvasTexture(lc);
  const s=new THREE.Sprite(new THREE.SpriteMaterial({{map:tex,transparent:true,depthWrite:false}}));
  s.scale.set(2.6,0.65,1);
  return s;
}}

// Tombstone shape
function makeTombstone(word, cause, idx){{
  const col = cause.includes("number")?0xe05555:cause.includes("punct")?0xe0a030:0x7070e0;
  const g=new THREE.Group();
  const body=new THREE.Mesh(
    new THREE.BoxGeometry(1.2,1.8,0.25),
    new THREE.MeshStandardMaterial({{color:0x334455,roughness:0.8,metalness:0.1}})
  );
  body.position.y=0.9; g.add(body);
  const top=new THREE.Mesh(
    new THREE.CylinderGeometry(0.6,0.6,0.25,12,1,false,0,Math.PI),
    new THREE.MeshStandardMaterial({{color:0x2a3a4a,roughness:0.9}})
  );
  top.rotation.z=Math.PI/2; top.position.set(0,1.92,0); g.add(top);
  const sp=makeSprite(word,col);
  sp.position.set(0,2.8,0); g.add(sp);
  const cols=8;
  const r=idx%cols, c=Math.floor(idx/cols);
  g.position.set(-14+r*2.2-8, 0, c*2.5-6);
  return g;
}}

// Pedestal / throne
function makePedestal(word, score, rank, idx){{
  const h=Math.max(0.4, score*18);
  const col=rank===0?0xffd700:rank===1?0xc0c0c0:rank===2?0xcd7f32:0x534AB7;
  const g=new THREE.Group();
  const ped=new THREE.Mesh(
    new THREE.BoxGeometry(1.5,h,1.5),
    new THREE.MeshStandardMaterial({{color:col,roughness:0.3,metalness:0.4,
      emissive:new THREE.Color(col),emissiveIntensity:0.15}})
  );
  ped.position.y=h/2; g.add(ped);
  if(rank<3){{
    const crown=new THREE.Mesh(
      new THREE.ConeGeometry(0.35,0.6,6),
      new THREE.MeshStandardMaterial({{color:0xffe066,metalness:0.8,roughness:0.2}})
    );
    crown.position.y=h+0.5; g.add(crown);
  }}
  const sp=makeSprite(word,rank<3?0xffe066:0xaaaaff);
  sp.position.set(0,h+1.1,0); g.add(sp);
  const cols=5;
  const r=idx%cols, c=Math.floor(idx/cols);
  g.position.set(14+r*3.2-8, 0, c*3.5-5);
  return g;
}}

// Populate graveyard
DEAD.slice(0,50).forEach((d,i)=>scene.add(makeTombstone(d[0],d[1],i)));

// Populate throne room
const maxS=Math.max(...ALIVE.map(a=>a.tfidf_score));
ALIVE.forEach((a,i)=>scene.add(makePedestal(a.word,a.tfidf_score/maxS,i,i)));

// Floating particles (left = red, right = blue)
const pGeo=new THREE.SphereGeometry(0.06,6,6);
const particles=[];
for(let i=0;i<40;i++){{
  const side=Math.random()<0.5?-1:1;
  const mat=new THREE.MeshStandardMaterial({{
    color:side<0?0xe05555:0x7c8dff,
    emissive:side<0?0xe05555:0x7c8dff,
    emissiveIntensity:0.6
  }});
  const p=new THREE.Mesh(pGeo,mat);
  p.position.set(side*(3+Math.random()*12),Math.random()*6,Math.random()*14-7);
  p.userData={{vy:0.005+Math.random()*0.01,side}};
  scene.add(p); particles.push(p);
}}

// Camera drag
let isDrag3=false,lmx=0,lmy=0,rotY3=0,camD3=32;
canvas3.addEventListener("mousedown",e=>{{isDrag3=true;lmx=e.clientX;lmy=e.clientY;canvas3.style.cursor="grabbing"}});
window.addEventListener("mouseup",()=>{{isDrag3=false;canvas3.style.cursor="grab"}});
window.addEventListener("mousemove",e=>{{
  if(!isDrag3)return;
  rotY3+=(e.clientX-lmx)*0.007;
  camD3=Math.max(10,Math.min(50,camD3-(e.clientY-lmy)*0.08));
  lmx=e.clientX;lmy=e.clientY;
}});
canvas3.addEventListener("wheel",e=>{{
  e.preventDefault();
  camD3=Math.max(10,Math.min(50,camD3+e.deltaY*0.025));
}},{{passive:false}});

let t3=0;
function animate3(){{
  requestAnimationFrame(animate3);
  t3+=0.012;
  if(!isDrag3) rotY3+=0.003;
  const el=Math.PI/10;
  camera.position.x=camD3*Math.cos(el)*Math.sin(rotY3);
  camera.position.y=camD3*Math.sin(el)+2;
  camera.position.z=camD3*Math.cos(el)*Math.cos(rotY3);
  camera.lookAt(0,0,0);
  particles.forEach(p=>{{
    p.position.y+=p.userData.vy;
    if(p.position.y>7)p.position.y=0;
    p.material.emissiveIntensity=0.3+Math.sin(t3+p.position.x)*0.4;
  }});
  renderer.render(scene,camera);
}}
animate3();
</script>
</body>
</html>"""

with open(DASHBOARD_OUT, "w", encoding="utf-8") as f:
    f.write(HTML_DASHBOARD)

print(f"\n✅ Dashboard saved → {DASHBOARD_OUT}")
print(f"   Open it in any browser — no install needed to VIEW it")



INNOVATION: GRAVEYARD & THRONE ROOM DASHBOARD

✅ Dashboard saved → nlp_dashboard.html
   Open it in any browser — no install needed to VIEW it


In [49]:



# ─────────────────────────────────────────────────────────────
#  CELL 11 — Q1(d) + ALL GRAPHS using Plotly only
#            (Required scatter + 6 extra graphs)
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Q1(d): ALL PLOTLY VISUALIZATIONS")
print("=" * 60)

COLORS = px.colors.qualitative.Vivid


# ── Graph 1 (REQUIRED): TF-IDF Scatter Plot ──────────────────
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=top_tfidf_df["word"],
    y=top_tfidf_df["tfidf_score"],
    mode="markers+text",
    text=top_tfidf_df["tfidf_score"].round(4),
    textposition="top center",
    marker=dict(
        size=top_tfidf_df["tfidf_score"] * 800 + 8,
        color=top_tfidf_df["tfidf_score"],
        colorscale="Viridis",
        showscale=True,
        colorbar=dict(title="TF-IDF Score"),
        line=dict(width=1, color="white")
    ),
    hovertemplate="<b>%{x}</b><br>TF-IDF: %{y:.5f}<extra></extra>"
))
fig1.update_layout(
    title=dict(text="📊 TF-IDF Scores — Top Words (Scatter Plot)", font_size=18),
    xaxis_title="Words",
    yaxis_title="Mean TF-IDF Score",
    xaxis_tickangle=-40,
    height=480,
    template="plotly_white",
)
fig1.show()
print("✅ Graph 1: TF-IDF Scatter — done")


# ── Graph 2: Preprocessing Funnel ────────────────────────────
funnel_labels  = list(stage_counts.keys())
funnel_values  = list(stage_counts.values())

fig2 = go.Figure(go.Funnel(
    y=funnel_labels,
    x=funnel_values,
    textinfo="value+percent initial",
    marker=dict(color=[
        "#378ADD","#1D9E75","#EF9F27","#D85A30",
        "#534AB7","#993556","#0F6E56","#BA7517","#639922"
    ][:len(funnel_labels)]),
    connector=dict(line=dict(color="royalblue", width=2))
))
fig2.update_layout(
    title=dict(text="🔽 Preprocessing Pipeline Funnel — Word Survival", font_size=18),
    height=500,
    template="plotly_white"
)
fig2.show()
print("✅ Graph 2: Preprocessing Funnel — done")


# ── Graph 3: Top Word Frequency Bar ──────────────────────────
top_freq = freq_df.head(TOP_N_WORDS)
fig3 = go.Figure(go.Bar(
    x=top_freq["word"],
    y=top_freq["frequency"],
    marker=dict(
        color=top_freq["frequency"],
        colorscale="Teal",
        showscale=True
    ),
    text=top_freq["frequency"],
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Frequency: %{y}<extra></extra>"
))
fig3.update_layout(
    title=dict(text=f"📈 Top {TOP_N_WORDS} Most Frequent Words", font_size=18),
    xaxis_title="Word",
    yaxis_title="Frequency",
    xaxis_tickangle=-40,
    height=460,
    template="plotly_white"
)
fig3.show()
print("✅ Graph 3: Word Frequency Bar — done")


# ── Graph 4: Zipf's Law Log-Log Scatter ──────────────────────
fig4 = go.Figure()
fig4.add_trace(go.Scatter(
    x=freq_df["log_rank"],
    y=freq_df["log_freq"],
    mode="markers",
    marker=dict(size=5, color="#534AB7", opacity=0.65),
    name="Observed",
    hovertemplate="Rank: %{customdata}<br>Freq: %{y:.2f}<extra></extra>",
    customdata=freq_df["rank"]
))
# ideal Zipf line
z = np.polyfit(freq_df["log_rank"], freq_df["log_freq"], 1)
p = np.poly1d(z)
fig4.add_trace(go.Scatter(
    x=freq_df["log_rank"],
    y=p(freq_df["log_rank"]),
    mode="lines",
    line=dict(color="#D85A30", width=2, dash="dash"),
    name=f"Zipf fit (slope={z[0]:.2f})"
))
fig4.update_layout(
    title=dict(text="📉 Zipf's Law — Log Rank vs Log Frequency", font_size=18),
    xaxis_title="Log₁₀ Rank",
    yaxis_title="Log₁₀ Frequency",
    height=460,
    template="plotly_white",
    legend=dict(x=0.75, y=0.95)
)
fig4.show()
print("✅ Graph 4: Zipf's Law — done")


# ── Graph 5: Top Bigrams Bar ──────────────────────────────────
fig5 = go.Figure(go.Bar(
    x=bigram_df["count"],
    y=bigram_df["bigram"],
    orientation="h",
    marker=dict(color=bigram_df["count"], colorscale="Sunset", showscale=True),
    text=bigram_df["count"],
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>Count: %{x}<extra></extra>"
))
fig5.update_layout(
    title=dict(text="🔗 Top 20 Bigrams (Two-Word Phrases)", font_size=18),
    xaxis_title="Frequency",
    yaxis_title="Bigram",
    height=560,
    template="plotly_white",
    yaxis=dict(autorange="reversed")
)
fig5.show()
print("✅ Graph 5: Bigrams — done")


# ── Graph 6: TF-IDF Heatmap by chunk ─────────────────────────
top20_features = list(mean_tfidf.head(20).index)
heatmap_data   = tfidf_df[top20_features].head(min(20, len(tfidf_df)))

fig6 = go.Figure(go.Heatmap(
    z=heatmap_data.values.round(4),
    x=top20_features,
    y=heatmap_data.index.tolist(),
    colorscale="Plasma",
    hoverongaps=False,
    hovertemplate="Chunk: %{y}<br>Word: %{x}<br>TF-IDF: %{z:.4f}<extra></extra>"
))
fig6.update_layout(
    title=dict(text="🌡️ TF-IDF Heatmap — Words × Document Chunks", font_size=18),
    xaxis_title="Top Words",
    yaxis_title="Document Chunk",
    xaxis_tickangle=-35,
    height=520,
    template="plotly_white"
)
fig6.show()
print("✅ Graph 6: TF-IDF Heatmap — done")


# ── Graph 7: Stemming vs Lemmatization Grouped Bar ───────────
if len(stem_lemma_df) > 0:
    fig7 = go.Figure()
    fig7.add_trace(go.Bar(
        name="Original length",
        x=stem_lemma_df["original"],
        y=stem_lemma_df["orig_len"],
        marker_color="#378ADD",
        text=stem_lemma_df["original"],
        textposition="inside"
    ))
    fig7.add_trace(go.Bar(
        name="Stemmed length",
        x=stem_lemma_df["original"],
        y=stem_lemma_df["stem_len"],
        marker_color="#D85A30",
        text=stem_lemma_df["stemmed"],
        textposition="inside"
    ))
    fig7.add_trace(go.Bar(
        name="Lemmatized length",
        x=stem_lemma_df["original"],
        y=stem_lemma_df["lemma_len"],
        marker_color="#1D9E75",
        text=stem_lemma_df["lemmatized"],
        textposition="inside"
    ))
    fig7.update_layout(
        barmode="group",
        title=dict(text="⚖️ Stemming vs Lemmatization — Word Length Comparison", font_size=18),
        xaxis_title="Original Word",
        yaxis_title="Word Length (chars)",
        xaxis_tickangle=-35,
        height=480,
        template="plotly_white",
        legend=dict(x=0.75, y=0.95)
    )
    fig7.show()
    print("✅ Graph 7: Stemming vs Lemmatization — done")


# ── Graph 8: Stopword vs Content Word Sunburst ───────────────
pos_tags = nltk.pos_tag(valid_words[:500])
pos_map  = {"NN":"Noun","NNS":"Noun","NNP":"Proper Noun","NNPS":"Proper Noun",
            "VB":"Verb","VBD":"Verb","VBG":"Verb","VBN":"Verb","VBP":"Verb","VBZ":"Verb",
            "JJ":"Adjective","JJR":"Adjective","JJS":"Adjective",
            "RB":"Adverb","RBR":"Adverb","RBS":"Adverb"}
pos_counts = collections.Counter()
for _, tag in pos_tags:
    pos_counts[pos_map.get(tag, "Other")] += 1

sunburst_labels  = ["All Words",
                    "Stop Words", "Content Words"] + list(pos_counts.keys())
sunburst_parents = ["",
                    "All Words", "All Words"] + ["Content Words"] * len(pos_counts)
sunburst_values  = [len(tokens),
                    len(stop_words_found), len(valid_words)] + list(pos_counts.values())

fig8 = go.Figure(go.Sunburst(
    labels=sunburst_labels,
    parents=sunburst_parents,
    values=sunburst_values,
    branchvalues="total",
    hovertemplate="<b>%{label}</b><br>Count: %{value}<br>%{percentParent:.1%} of parent<extra></extra>",
    marker=dict(colors=[
        "#f0f0f0","#D85A30","#1D9E75",
        "#378ADD","#534AB7","#EF9F27","#993556","#0F6E56","#BA7517"
    ][:len(sunburst_labels)])
))
fig8.update_layout(
    title=dict(text="🌞 Word Composition Sunburst — Stopwords vs POS Categories", font_size=18),
    height=520,
    template="plotly_white"
)
fig8.show()
print("✅ Graph 8: Sunburst — done")


# ── Graph 9: One Hot Encoding Heatmap ────────────────────────
ohe_display = ohe_df.set_index("Word").head(25)
fig9 = go.Figure(go.Heatmap(
    z=ohe_display.values,
    x=ohe_display.columns.tolist(),
    y=ohe_display.index.tolist(),
    colorscale=[[0,"#f5f5f5"],[1,"#534AB7"]],
    showscale=False,
    hovertemplate="Word: %{y}<br>Vocab: %{x}<br>OHE: %{z}<extra></extra>"
))
fig9.update_layout(
    title=dict(text="🔢 One Hot Encoding Matrix (25 words × 30 vocab)", font_size=18),
    xaxis_title="Vocabulary",
    yaxis_title="Word",
    xaxis_tickangle=-45,
    height=540,
    template="plotly_white"
)
fig9.show()
print("✅ Graph 9: OHE Heatmap — done")


# ── Graph 10: Top Trigrams ────────────────────────────────────
fig10 = go.Figure(go.Bar(
    x=trigram_df["count"],
    y=trigram_df["trigram"],
    orientation="h",
    marker=dict(color=trigram_df["count"], colorscale="Tealgrn", showscale=True),
    text=trigram_df["count"],
    textposition="outside",
))
fig10.update_layout(
    title=dict(text="🔗 Top 15 Trigrams (Three-Word Phrases)", font_size=18),
    xaxis_title="Frequency",
    yaxis_title="Trigram",
    height=500,
    template="plotly_white",
    yaxis=dict(autorange="reversed")
)
fig10.show()
print("✅ Graph 10: Trigrams — done")







Q1(d): ALL PLOTLY VISUALIZATIONS


✅ Graph 1: TF-IDF Scatter — done


✅ Graph 2: Preprocessing Funnel — done


✅ Graph 3: Word Frequency Bar — done


✅ Graph 4: Zipf's Law — done


...

✅ Graph 6: TF-IDF Heatmap — done


✅ Graph 7: Stemming vs Lemmatization — done


✅ Graph 8: Sunburst — done


✅ Graph 9: OHE Heatmap — done


✅ Graph 10: Trigrams — done


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELL 13 — Final Summary
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("COMPLETE PIPELINE SUMMARY")
print("=" * 60)
print(f"""
✅  Q1(a)  PDF reading          — {total_pages} pages extracted
✅  Q1(b)  Preprocessing        — 9 steps with regex shown
           Regex 1: {REGEX_NUMBERS}  → removed numbers
           Regex 2: {REGEX_SYMBOLS}  → removed symbols
           Regex 3: {REGEX_SPACES}          → removed extra spaces
           Stop words found     : {len(stop_words_found):,}
           Valid words kept     : {len(valid_words):,}
           Stemming done        : {len(stemmed_words):,} words
           Lemmatization done   : {len(lemmatized_words):,} words
✅  Q1(c)  OHE                  — shape {ohe_df.shape}
✅  Q1(c)  TF-IDF               — shape {tfidf_df.shape}
✅  Q1(d)  Graphs (Plotly only)
           Graph 1  — TF-IDF Scatter (REQUIRED)
           Graph 2  — Preprocessing Funnel
           Graph 3  — Word Frequency Bar
           Graph 4  — Zipf's Law Log-Log
           Graph 5  — Top 20 Bigrams
           Graph 6  — TF-IDF Heatmap
           Graph 7  — Stemming vs Lemmatization
           Graph 8  — Sunburst (POS + Stopword)
           Graph 9  — OHE Heatmap
           Graph 10 — Top 15 Trigrams
✅  INNOVATION
           Language detected    : {detected_lang}
           Vocab richness       : {vocab_richness}%
           3D Dashboard         : {DASHBOARD_OUT}
           Graveyard tokens     : {len(dead_tokens)}
           Throne room tokens   : {len(alive_tokens)}
""")
print("=" * 60)

print("=" * 60)


COMPLETE PIPELINE SUMMARY

✅  Q1(a)  PDF reading          — 12 pages extracted
✅  Q1(b)  Preprocessing        — 9 steps with regex shown
           Regex 1: \d+  → removed numbers
           Regex 2: [^a-z\s]  → removed symbols
           Regex 3: \s+          → removed extra spaces
           Stop words found     : 635
           Valid words kept     : 1,631
           Stemming done        : 1,631 words
           Lemmatization done   : 1,631 words
✅  Q1(c)  OHE                  — shape (50, 31)
✅  Q1(c)  TF-IDF               — shape (6, 100)
✅  Q1(d)  Graphs (Plotly only)
           Graph 1  — TF-IDF Scatter (REQUIRED)
           Graph 2  — Preprocessing Funnel
           Graph 3  — Word Frequency Bar
           Graph 4  — Zipf's Law Log-Log
           Graph 5  — Top 20 Bigrams
           Graph 6  — TF-IDF Heatmap
           Graph 7  — Stemming vs Lemmatization
           Graph 8  — Sunburst (POS + Stopword)
           Graph 9  — OHE Heatmap
           Graph 10 — Top 15 Trigrams
✅  